# NB_hospital_daily_summary
Builds `<catalog>.gold.hospital_daily_summary` per STTM *Silver to Gold (CareSync Health Network) v1.0*.

* **Grain:** one row per hospital per calendar day (days with at least one lab test)
* **Driving table:** `silver.lab_results` grouped by hospital_id, test_date
* **Joins:** LEFT JOIN `silver.hospitals` on hospital_id; LEFT JOIN `silver.calendar` on `datekey = int(date_format(test_date, 'yyyyMMdd'))` (STTM calls this `date_key`; the actual column is `datekey` INT)
* **Window:** LAG(total_tests) partitioned by hospital_id ordered by summary_date (previous recorded day)
* **Filters:** none
* **Load:** FULL rebuild - CREATE OR REPLACE TABLE ... AS SELECT

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE databeli_adb.gold.hospital_daily_summary
COMMENT 'Gold: one row per hospital per day with lab test volume, abnormal/critical rates and day-over-day trend. Full rebuild from silver.'
AS
WITH daily AS (
  SELECT hospital_id,
         test_date                            AS summary_date,
         COUNT(*)                             AS total_tests,
         COUNT(DISTINCT patient_id)           AS unique_patients_tested,
         COUNT(DISTINCT doctor_id)            AS active_doctor_count,
         COUNT_IF(result_status = 'Abnormal') AS abnormal_test_count,
         COUNT_IF(result_status = 'Critical') AS critical_test_count
  FROM databeli_adb.silver.lab_results
  GROUP BY hospital_id, test_date
),
with_lag AS (
  SELECT d.*,
         LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests
  FROM daily d
)
SELECT
  CAST(w.hospital_id AS INT)             AS hospital_id,
  CAST(h.hospital_name AS STRING)        AS hospital_name,
  CAST(w.summary_date AS DATE)           AS summary_date,
  CAST(c.day_name AS STRING)             AS day_name,
  CAST(c.is_weekend AS BOOLEAN)          AS is_weekend,
  CAST(w.total_tests AS INT)             AS total_tests,
  CAST(w.unique_patients_tested AS INT)  AS unique_patients_tested,
  CAST(w.active_doctor_count AS INT)     AS active_doctor_count,
  CAST(w.abnormal_test_count AS INT)     AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)     AS critical_test_count,
  CAST(ROUND(w.abnormal_test_count * 100.0 / NULLIF(w.total_tests, 0), 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
  CAST(ROUND(w.critical_test_count * 100.0 / NULLIF(w.total_tests, 0), 2) AS DECIMAL(5,2)) AS critical_rate_pct,
  CAST(w.prev_day_total_tests AS INT)    AS prev_day_total_tests,
  CAST(ROUND((w.total_tests - w.prev_day_total_tests) * 100.0 / NULLIF(w.prev_day_total_tests, 0), 2) AS DECIMAL(6,2)) AS test_volume_change_pct
FROM with_lag w
LEFT JOIN databeli_adb.silver.hospitals h
       ON w.hospital_id = h.hospital_id
LEFT JOIN databeli_adb.silver.calendar c
       ON c.datekey = CAST(date_format(w.summary_date, 'yyyyMMdd') AS INT)
""")

In [0]:
updated_df = spark.read.table("databeli_adb.gold.hospital_daily_summary")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))